# OpenMuse en Kaggle con Persistencia Real

Este notebook permite correr OpenMuse en Kaggle y **continuar exactamente donde lo dejaste** aunque se caiga la sesión.

- Guardado automático cada 5 minutos
- Restauración automática al reiniciar
- 100% Kaggle (sin Google Drive ni ngrok)

In [ ]:
# ====================== CELDA 1: Sistema de Persistencia ======================
import sys
import os

# Clonar este repo de persistencia si no existe
if not os.path.exists('/kaggle/working/openmuse-kaggle-persistence'):
    !git clone https://github.com/enrrutador/openmuse-kaggle-persistence.git /kaggle/working/openmuse-kaggle-persistence

sys.path.insert(0, '/kaggle/working/openmuse-kaggle-persistence')

from persistence.state import OpenMuseStateManager
from persistence.auto_save import start_auto_save

manager = OpenMuseStateManager()
manager.restore()

# Iniciar guardado automático cada 5 minutos
start_auto_save(manager, interval_minutes=5)

print('\n=== Sistema de persistencia listo ===')
print(f'DATA_DIR : {manager.get_data_dir()}')
print(f'Estado   : {"existe" if manager.exists() else "nuevo"}')

In [ ]:
# ====================== CELDA 2: Instalar OpenMuse ======================
!bash /kaggle/working/openmuse-kaggle-persistence/scripts/setup_openmuse.sh

In [ ]:
# ====================== CELDA 3: Configurar variables de entorno ======================
import os
from pathlib import Path

data_dir = str(manager.get_data_dir())

# Variables mínimas para que OpenMuse arranque en modo sample
os.environ['DATA_DIR'] = data_dir
os.environ['WORKSPACE_MODE'] = 'sample'
os.environ['AGENT_BACKEND'] = 'sample'
os.environ['PORT'] = '8787'
os.environ['HOST'] = '0.0.0.0'
os.environ['PUBLIC_API_URL'] = 'http://localhost:8787'
os.environ['TASK_WORKER_ENABLED'] = 'true'
os.environ['WEB_SEARCH_ENABLED'] = 'true'
os.environ['COMPUTER_ENABLED'] = 'false'

# Crear .env dentro de openmuse
env_content = f"""DATA_DIR={data_dir}
WORKSPACE_MODE=sample
AGENT_BACKEND=sample
PORT=8787
HOST=0.0.0.0
PUBLIC_API_URL=http://localhost:8787
TASK_WORKER_ENABLED=true
WEB_SEARCH_ENABLED=true
COMPUTER_ENABLED=false
"""

env_path = Path('/kaggle/working/openmuse/.env')
env_path.write_text(env_content)

print('Variables de entorno configuradas')
print(f'DATA_DIR = {data_dir}')

In [ ]:
# ====================== CELDA 4: Arrancar OpenMuse ======================
import subprocess
import time

print('Iniciando servidor de OpenMuse...')
print('(Esto puede tardar la primera vez)')

# Arrancar en background
process = subprocess.Popen(
    ['pnpm', 'dev'],
    cwd='/kaggle/working/openmuse',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy()
)

# Esperar un poco y mostrar logs iniciales
time.sleep(8)

print('\n--- Logs iniciales ---')
# Leer algunas líneas de salida
for _ in range(30):
    line = process.stdout.readline()
    if line:
        print(line.rstrip())
    else:
        break

print('\nServidor iniciado (proceso en background)')
print('El estado se sigue guardando automáticamente cada 5 minutos.')

## Notas importantes

- El estado se guarda solo en `/kaggle/working`. Para que sobreviva a un reinicio completo del notebook hacé **Save Version**.
- No estamos usando ngrok. La forma de acceder desde el iPhone se definirá más adelante.
- Si el notebook se cae, simplemente volvé a ejecutar las celdas en orden.